<a href="https://colab.research.google.com/github/qudwo9969-glitch/maritime-data-mining/blob/main/%ED%95%B4%EC%82%AC%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%A7%88%EC%9D%B4%EB%8B%9D%2011%EB%B2%88%EC%A7%B8%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd

import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

DATA_DIR = Path('/content/drive/MyDrive/type3_week13')  # 필요시 수정

required_files = [
    'churn_logit1.csv',
    'loan_default_logit2.csv'
]

print("=== 파일 확인 ===")
for f in required_files:
    file_path = DATA_DIR / f
    print(f"{f}: {'존재함' if file_path.exists() else '없음'}")

def logistic_analysis_report(model, df, y_col, threshold=0.5):
    result_table = pd.DataFrame({
        'coef': model.params,
        'p_value': model.pvalues,
        'odds_ratio': np.exp(model.params)
    })

    conf = model.conf_int()
    conf.columns = ['2.5%', '97.5%']
    conf['OR_2.5%'] = np.exp(conf['2.5%'])
    conf['OR_97.5%'] = np.exp(conf['97.5%'])

    df_result = df.copy()
    df_result['pred_prob'] = model.predict(df_result)
    df_result['pred_class'] = (df_result['pred_prob'] >= threshold).astype(int)

    acc = accuracy_score(df_result[y_col], df_result['pred_class'])
    err = 1 - acc
    cm = confusion_matrix(df_result[y_col], df_result['pred_class'])

    llf = model.llf
    llnull = model.llnull
    pseudo_r2 = model.prsquared
    residual_deviance = -2 * llf

    print("=== 회귀 요약표 ===")
    print(model.summary())

    print("=== 계수 / p-value / 오즈비 ===")
    print(result_table)

    print("=== 95% 신뢰구간 및 오즈비 신뢰구간 ===")
    print(conf)

    print("=== 모형 적합도 ===")
    print(f"log-likelihood      : {llf:.4f}")
    print(f"null log-likelihood : {llnull:.4f}")
    print(f"pseudo R^2          : {pseudo_r2:.4f}")
    print(f"residual deviance   : {residual_deviance:.4f}")

    print("=== 분류 성능 ===")
    print(f"accuracy   : {acc:.4f}")
    print(f"error rate : {err:.4f}")

    print("=== confusion matrix ===")
    print(cm)

    print("=== 예측확률 상위 10개 ===")
    print(df_result[[y_col, 'pred_prob', 'pred_class']].head(10))

    return result_table, conf, df_result, acc, err, residual_deviance

Mounted at /content/drive
=== 파일 확인 ===
churn_logit1.csv: 없음
loan_default_logit2.csv: 없음


In [12]:
# churn_logit1.csv 불러오기 및 데이터 확인

churn_df = pd.read_csv(DATA_DIR / 'churn_logit1.csv')

print("=== 데이터 상위 5행 ===")
display(churn_df.head())

print("=== 데이터 구조 ===")
print(churn_df.info())

print("=== 기술통계 ===")
display(churn_df.describe())

print("=== 종속변수 분포 ===")
display(churn_df['churn'].value_counts().sort_index())

  # 로지스틱 회귀 적합: churn ~ age + usage_hour + complaint_cnt

churn_model = smf.logit(
    formula='churn ~ age + usage_hour + complaint_cnt',
    data=churn_df
).fit()

churn_result_table, churn_conf, churn_pred_df, churn_acc, churn_err, churn_dev = logistic_analysis_report(
    churn_model, churn_df, 'churn'
)

      # 변수별 해석 문장 자동 출력

print("=== 변수별 해석 ===")
for var in ['age', 'usage_hour', 'complaint_cnt']:
    coef = churn_model.params[var]
    pval = churn_model.pvalues[var]
    odds = np.exp(coef)

    direction = "증가" if coef > 0 else "감소"

    print(f"\n[{var}]")
    print(f"- 계수: {coef:.4f}")
    print(f"- p-value: {pval:.6f}")
    print(f"- 오즈비: {odds:.4f}")

    if pval < 0.05:
        print(f"→ {var}는 유의한 변수이다.")
    else:
        print(f"→ {var}는 유의하다고 보기 어렵다.")

    print(f"→ {var}가 1단위 증가할 때 이탈 odds는 약 {odds:.4f}배가 되며, 방향은 {direction}이다.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/type3_week13/churn_logit1.csv'

In [ ]:
# loan_default_logit2.csv 불러오기 및 데이터 확인

loan_df = pd.read_csv(DATA_DIR / 'loan_default_logit2.csv')

print("=== 원본 데이터 상위 5행 ===")
display(loan_df.head())

print("=== 데이터 구조 ===")
print(loan_df.info())

print("=== 기술통계 ===")
display(loan_df.describe())

print("=== 종속변수 분포 ===")
display(loan_df['default'].value_counts().sort_index())

      # 'default'는 Python 예약어와 헷갈릴 수 있으므로 안전하게 이름 변경
loan_df = loan_df.rename(columns={'default': 'default_flag'})

print("변경된 컬럼명:")
print(loan_df.columns.tolist())

 # 로지스틱 회귀 적합: default_flag ~ income + debt_ratio + late_cnt

loan_model = smf.logit(
    formula='default_flag ~ income + debt_ratio + late_cnt',
    data=loan_df
).fit()

loan_result_table, loan_conf, loan_pred_df, loan_acc, loan_err, loan_dev = logistic_analysis_report(
    loan_model, loan_df, 'default_flag'
)

      # 변수별 해석 문장 자동 출력

print("=== 변수별 해석 ===")
for var in ['income', 'debt_ratio', 'late_cnt']:
    coef = loan_model.params[var]
    pval = loan_model.pvalues[var]
    odds = np.exp(coef)

    direction = "증가" if coef > 0 else "감소"

    print(f"\n[{var}]")
    print(f"- 계수: {coef:.6f}")
    print(f"- p-value: {pval:.6f}")
    print(f"- 오즈비: {odds:.6f}")

    if pval < 0.05:
        print(f"→ {var}는 유의한 변수이다.")
    else:
        print(f"→ {var}는 유의하다고 보기 어렵다.")

    print(f"→ {var}가 1단위 증가할 때 연체 odds는 약 {odds:.6f}배가 되며, 방향은 {direction}이다.")

      # 두 실습 결과 핵심 지표 비교

summary_df = pd.DataFrame({
    'dataset': ['churn_logit1.csv', 'loan_default_logit2.csv'],
    'accuracy': [churn_acc, loan_acc],
    'error_rate': [churn_err, loan_err],
    'log_likelihood': [churn_model.llf, loan_model.llf],
    'pseudo_R2': [churn_model.prsquared, loan_model.prsquared],
    'residual_deviance': [churn_dev, loan_dev]
})

print("=== 두 모형 비교표 ===")
display(summary_df)

4. 보고서 작성 템플릿 해답

[실습 1 보고서 템플릿 - 고객 이탈 분석]본 연구에서는 고객 이탈 여부를 설명하기 위해 로지스틱 회귀분석을 수행하였다.분석 결과, usage_hour(월 이용시간)와 complaint_cnt(최근 불만 건수) 변수는 유의한 영향을 보였고($p < 0.05$), 특히 complaint_cnt 변수의 오즈비는 1보다 큰 값(예: 1.5000)으로 나타나 해당 변수가 1단위 증가할 때 이탈 odds가 약 1.5배 변한다고 해석할 수 있다. 반면 age(고객 나이) 변수는 유의하지 않아 이탈에 직접적인 영향을 준다고 보기 어렵다. 또한 모형의 accuracy는 (실제 분석 결과 수치), error rate는 (1 - accuracy 수치)로 나타났다.




[실습 2 보고서 템플릿 - 대출 연체 분석]
본 연구에서는 대출 연체 여부를 설명하기 위해 로지스틱 회귀분석을 수행하였다.

분석 결과, income(소득), debt_ratio(부채비율), late_cnt(과거 연체 횟수) 변수는 모두 유의한 영향을 보였다. 특히 debt_ratio(또는 late_cnt) 변수의 오즈비가 크게 나타나 해당 변수의 증가가 연체 발생 odds를 크게 높이는 것으로 해석되었다. 모형의 log-likelihood는 (결과 수치), residual deviance는 (결과 수치)이며, accuracy는 (결과 수치), error rate는 (결과 수치)로 나타났다.



5. 추가 과제 해답



Q1. 예측 기준값(threshold)을 0.5가 아니라 0.4 또는 0.6으로 바꾸면 accuracy는 어떻게 달라지는가?
답변: 데이터의 분포에 따라 정확도(Accuracy)는 올라갈 수도 있고 내려갈 수도 있습니다. * 이유: 임계값(Threshold)을 변경하면 모델이 1(이탈/연체)이라고 판단하는 기준이 엄격해지거나 느슨해집니다. 만약 데이터에 1보다 0이 훨씬 많다면, 임계값을 0.6으로 높였을 때 0을 더 잘 맞추게 되어 전체적인 정확도가 올라갈 수 있습니다. 반대로 균형이 깨지면 정확도가 떨어질 수도 있습니다. 실무에서는 정확도뿐만 아니라 '연체자를 얼마나 놓치지 않고 잡아내는가(재현율)' 등도 함께 고려해야 합니다.




Q2. churn 데이터에서 가장 영향력이 큰 변수는 무엇인가?답변: 오즈비($exp(coef)$)가 1에서 가장 멀리 떨어진 변수입니다.이유: 로지스틱 회귀에서 변수의 영향력은 오즈비로 판단합니다. 가이드라인을 보면 complaint_cnt는 양(+)의 계수(오즈비 > 1)로 이탈을 증가시키고, usage_hour는 음(-)의 계수(오즈비 < 1)로 이탈을 감소시킵니다. 1을 기준으로 배수가 가장 크게 뛰거나(예: 2.5배), 가장 크게 줄어드는(예: 0.2배) 변수가 영향력이 가장 큰 변수입니다. 일반적으로 complaint_cnt나 usage_hour가 유의미한 주 원인이 됩니다.





Q3. loan default 데이터에서 실무적으로 가장 중요한 관리 변수는 무엇인가?
답변: debt_ratio(부채비율) 또는 late_cnt(과거 연체 횟수) 입니다.

이유: 실습 가이드에서 "debt_ratio의 오즈비가 매우 크게 나타나 부채비율이 높을수록 연체 발생 odds가 크게 증가한다"고 언급되어 있습니다. 오즈비가 크다는 것은 리스크를 급격하게 높이는 핵심 요인이라는 뜻이므로, 은행 실무에서는 대출 승인 시 부채비율의 상한선을 엄격하게 제한하거나 과거 연체 횟수가 있는 고객을 중점 관리해야 합니다





Q4. 오즈비가 1보다 작은 경우와 1보다 큰 경우를 각각 실제 문장으로 써 보라.
오즈비가 1보다 큰 경우 (예: complaint_cnt 오즈비가 1.58일 때)

"최근 불만 건수가 1건 증가할 때마다, 고객이 서비스를 이탈할 odds(확률/1-확률)는 약 1.58배 증가합니다."

오즈비가 1보다 작은 경우 (예: usage_hour 오즈비가 0.85일 때)

"월 이용시간이 1시간 증가할 때마다, 고객이 서비스를 이탈할 odds는 약 0.85배로 감소합니다. (즉, 이탈 위험이 약 15% 감소합니다.)"







